## индивидуальная задача 4 (вариант 9) 

Класс «Сотрудник» (Employee)
Свойства:

name (имя)
position (должность)
salary (зарплата, от 0 до 100000)
experience (опыт работы в годах, целое неотрицательное)
skills (навыки): список строк

Методы:
promote(new_position, salary_increase): изменить должность на new_position и увеличить зарплату на salary_increase. Зарплата не может превышать 100000.
add_skill(skill): добавить навык в список skills 
get_bonus(percentage): рассчитать и вернуть бонус в процентах от зарплаты (percentage).
work(): выводит в консоль фразу "Employee {name} is working on {position}."
random_training(): случайным образом (с вероятностью 40%) увеличивает experience на 1 год, а также с вероятностью 20% добавляет случайный навык из заранее заданного списка (например, "Python", "SQL", "Project Management").
collaborate(other_employee): если у текущего сотрудника и other_employee есть хотя бы один общий навык, то оба получают повышение зарплаты на 5% (но не выше 100000) и возвращается строка "Collaboration successful". Если общих навыков нет – ничего не меняется, возвращается "No common skills".

Сотрудника можно «сложить» (+) с числом: зарплата увеличивается на это число (но не выше 100000).
Вызов сотрудника с числом (процент премии) возвращает строку: "Bonus for {name}: {bonus} rub.", где bonus – рассчитанная сумма (зарплата * процент / 100).
Сотрудников можно сравнивать между собой: сначала по зарплате (по убыванию), затем по опыту (по убыванию), затем по имени (по алфавиту).

Класс-наследник: «Менеджер» (Manager)
Дополнительные свойства:
team (список сотрудников в команде, по умолчанию пустой).
team_performance (эффективность команды, число от 0 до 100, вычисляется как средний опыт членов команды, но не более 100).

Дополнительные методы:

add_to_team(employee): добавляет сотрудника в команду менеджера 
conduct_meeting(): каждый член команды получает случайный навык из списка навыков менеджера, а сам менеджер увеличивает зарплату на 2% (но не выше 100000).
Переопределить collaborate(other_employee): если other_employee – тоже менеджер, то при успешной коллаборации обе команды объединяются (списки сотрудников сливаются без дубликатов).

Обязательно использование конструктора, декораторов  и метода __str__.

In [18]:
class Employee:
    RANDOM_SKILLS = ["Python", "SQL", "Project Management"]

    def __init__(self, name: str, position: str, salary: float, experience: int, skills=None):
        self.name = name
        self.position = position
        self.salary = max(0, min(salary, 100000))
        self.experience = max(0, experience)
        self.skills = skills if skills is not None else []

    def promote(self, new_position: str, salary_increase: float):
        self.position = new_position
        self.salary = min(self.salary + salary_increase, 100000)

    def add_skill(self, skill: str):
        self.skills.append(skill)

    def get_bonus(self, percentage: float) -> float:
        return self.salary * percentage / 100

    def work(self):
        print(f"Employee {self.name} is working on {self.position}.")

    def random_training(self):
        if random.random() < 0.4:
            self.experience += 1
        if random.random() < 0.2:
            skill = random.choice(self.RANDOM_SKILLS)
            self.add_skill(skill)

    def collaborate(self, other_employee: 'Employee') -> str:
        common = set(self.skills) & set(other_employee.skills)
        if common:
            self.salary = min(self.salary * 1.05, 100000)
            other_employee.salary = min(other_employee.salary * 1.05, 100000)
            return "Collaboration successful"
        else:
            return "No common skills"

    def __add__(self, other):
        if isinstance(other, (int, float)):
            new_salary = min(self.salary + other, 100000)
            new_emp = Employee(self.name, self.position, new_salary, self.experience, self.skills.copy())
            return new_emp
        return NotImplemented

    def __call__(self, percentage: float) -> str:
        bonus = self.get_bonus(percentage)
        return f"Bonus for {self.name}: {bonus} rub."

    def __lt__(self, other):
        if not isinstance(other, Employee):
            return NotImplemented
        return (-self.salary, -self.experience, self.name) < (-other.salary, -other.experience, other.name)

    def __str__(self):
        return f"{self.name} ({self.position}) — зарплата: {self.salary} руб., опыт: {self.experience} лет, " \
            f"навыки: {', '.join(self.skills) if self.skills else 'нет'}"

    
class Manager(Employee):
    
    def __init__(self, name: str, position: str, salary: float, experience: int, skills=None, team=None):
        super().__init__(name, position, salary, experience, skills)
        self.team = team if team is not None else []

    def team_performance(self) -> float:
        if not self.team:
            return 0.0
        avg_exp = sum(emp.experience for emp in self.team) / len(self.team)
        return min(avg_exp, 100.0)

    def add_to_team(self, employee: Employee):
        self.team.append(employee)

    def conduct_meeting(self):
        for emp in self.team:
            skill = random.choice(self.skills)
            emp.add_skill(skill)
        self.salary = min(self.salary * 1.02, 100000)

    def collaborate(self, other_employee):
        result = super().collaborate(other_employee)
        if result == "Collaboration successful" and isinstance(other_employee, Manager):
            new_team = self.team.copy()
            for emp in other_employee.team:
                if emp not in new_team:
                    new_team.append(emp)
            self.team = new_team
            other_employee.team = new_team.copy()
        return result
        
emp = Employee("Иван", "Разработчик", 80000, 5, ["Python"])
print(emp)
mng = Manager("Николай", "Менеджер", 70000, 3, ["Project Management"])
print(mng)

Иван (Разработчик) — зарплата: 80000 руб., опыт: 5 лет, навыки: Python
Николай (Менеджер) — зарплата: 70000 руб., опыт: 3 лет, навыки: Project Management
